# NLP on Steroids 💉 — Classic NLP on the GPU with NVIDIA RAPIDS

**Module 03 · Notebook 02 · NCA-GENL Bootcamp**

In notebook 01 we learned the classic NLP flow: *preprocessing → vectorization → classification*.
In this notebook we run the **exact same flow** — on the **GPU**, using the **NVIDIA RAPIDS** stack:

| Library | Role |
|---|---|
| **cuDF** | GPU version of pandas (DataFrame) |
| **nvtext** | GPU text ops inside cuDF (tokenize, n-gram, minhash) |
| **cuML** | GPU version of scikit-learn (TF-IDF, Logistic Regression, etc.) |

> ⚠️ **GPU required**: Runtime → Change runtime type → **T4 GPU** before running the cells below.

**What you will master:**
1. Running your existing pandas/sklearn code on the GPU **without changing a single line**
2. Preprocessing ±150k Wikipedia paragraphs in seconds
3. Near-duplicate deduplication with MinHash — a real step in LLM corpus preparation
4. TF-IDF + sentiment classification with the native cuML API

In [ ]:
# Check the GPU — you should see a Tesla T4
!nvidia-smi

In [ ]:
# RAPIDS comes pre-installed on Colab GPU runtimes — just import it.
# If this cell fails, run the fallback cell below.
import cudf, cuml
print("cuDF :", cudf.__version__)
print("cuML :", cuml.__version__)

In [ ]:
# FALLBACK — run this cell ONLY if the import above failed (takes ±5 min).
# Remove the hash on the line below, then run:
# !pip install -q --extra-index-url=https://pypi.nvidia.com cudf-cu12 cuml-cu12
print("If the cudf/cuml import above SUCCEEDED, skip this cell.")

## Act 1 — The Magic Trick: Old Code, New GPU 🎩

NVIDIA's claim: the pandas + sklearn code you already wrote can run on the GPU with **zero code changes**, via two "accelerators":

- `cudf.pandas` — hijacks `import pandas`; every operation is routed to cuDF (GPU); unsupported ops automatically *fall back* to real pandas.
- `cuml.accel` — the same idea for scikit-learn estimators → cuML.

**The trick is not changing the code, but changing how you RUN it.** Because the accelerator must be active *before* `import pandas`, we write the pipeline as a script and run it twice: normally (CPU) and through the accelerator (GPU).

Data: **SmSA** from the IndoNLU benchmark — Indonesian reviews labeled with sentiment (positive/negative/neutral), the same dataset mentioned in notebook 01.

In [ ]:
# Download SmSA (URL pinned to a commit for stability)
!wget -q "https://raw.githubusercontent.com/IndoNLP/indonlu/ce728f6926a36174b9923dfe49d6a6839b6e9bb7/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv" -O smsa_train.tsv
!wget -q "https://raw.githubusercontent.com/IndoNLP/indonlu/ce728f6926a36174b9923dfe49d6a6839b6e9bb7/dataset/smsa_doc-sentiment-prosa/valid_preprocess.tsv" -O smsa_valid.tsv
!wc -l smsa_train.tsv smsa_valid.tsv

In [ ]:
%%writefile pipeline.py
# The classic notebook-01 pipeline: TF-IDF + Logistic Regression.
# There is NOT a single line of GPU code in this file.
import time
t_all = time.perf_counter()
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

train = pd.read_csv("smsa_train.tsv", sep="\t", names=["text", "label"])
valid = pd.read_csv("smsa_valid.tsv", sep="\t", names=["text", "label"])
# Duplicate train 20x: simulates a production-scale workload (~220k documents)
train = pd.concat([train] * 20, ignore_index=True)

t0 = time.perf_counter()
vec = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2))
Xtr = vec.fit_transform(train.text)
Xva = vec.transform(valid.text)
t_vec = time.perf_counter() - t0

t0 = time.perf_counter()
clf = LogisticRegression(max_iter=200)
clf.fit(Xtr, train.label)
acc = accuracy_score(valid.label, clf.predict(Xva))
t_fit = time.perf_counter() - t0
print(f"accuracy={acc:.4f}  vectorize={t_vec:.1f}s  fit+predict={t_fit:.1f}s  total={time.perf_counter()-t_all:.1f}s")

In [ ]:
# Baseline: pure CPU (plain pandas + sklearn)
!python pipeline.py

In [ ]:
# The magic: the SAME file, run through the GPU accelerators
!python -m cudf.pandas -m cuml.accel pipeline.py

### What just happened?

- **Same accuracy** → the computation is statistically identical.
- **Very different time** → the pandas parts (load, concat, strings) ran on cuDF; supported sklearn estimators ran on cuML.
- Operations not yet supported on GPU automatically **fall back to CPU** — your code never crashes because of this.

> 💡 **When is this trick worth it?** When the data is big enough. For small data, CPU↔GPU transfer costs more than the computation itself — the GPU can actually be slower. This is a universal pattern in GPU computing.

## Act 2 — Dissecting nvtext: Wikipedia-Scale Preprocessing 🔬

Magic over; now we use the GPU API **deliberately**. cuDF has a text module (*nvtext*) with operations pandas **simply does not have**: `tokenize`, `token_count`, `ngrams_tokenize`, `minhash`.

We also upgrade the corpus: **Indonesian Wikipedia** — we take ±150k paragraphs via streaming (no full-dump download).

In [ ]:
!pip install -q datasets

from datasets import load_dataset

# Streaming: take the first 20k articles, split into paragraphs
ds = load_dataset("wikimedia/wikipedia", "20231101.id", split="train", streaming=True)
paragraphs = []
for i, art in enumerate(ds):
    if i >= 20_000:
        break
    paragraphs += [p for p in art["text"].split("\n") if len(p) > 80]

print(f"{len(paragraphs):,} paragraphs collected")

In [ ]:
import time
import pandas as pd
import cudf

pdf = pd.Series(paragraphs, name="text")     # CPU
gdf = cudf.Series(paragraphs, name="text")   # GPU

BENCH = {}  # hasil benchmark, dipakai untuk ringkasan & slide

def bench(group, label, fn):
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    BENCH.setdefault(group, {})[label] = dt
    print(f"{group:12s} | {label:6s} | {dt:7.3f} s")
    return out

In [ ]:
# --- Tokenization + token count ---
n_cpu = bench("tokenize", "CPU", lambda: pdf.str.split().map(len).sum())
n_gpu = bench("tokenize", "GPU", lambda: gdf.str.token_count().sum())
print(f"total tokens: CPU={n_cpu:,} GPU={n_gpu:,}")

In [ ]:
# --- Word frequency (top-15) ---
top_cpu = bench("wordcount", "CPU", lambda: pdf.str.lower().str.split().explode().value_counts().head(15))
top_gpu = bench("wordcount", "GPU", lambda: gdf.str.lower().str.tokenize().value_counts().head(15))
print(top_gpu)

In [ ]:
# --- Bigrams (n-grams with n=2) ---
def bigram_cpu():
    toks = pdf.str.lower().str.split()
    return toks.map(lambda t: ["_".join(p) for p in zip(t, t[1:])]).explode().value_counts().head(10)

def bigram_gpu():
    return gdf.str.lower().str.ngrams_tokenize(2, separator="_").value_counts().head(10)

bench("bigram", "CPU", bigram_cpu)
big = bench("bigram", "GPU", bigram_gpu)
print(big)

### Halftime score

Run the cell below for a recap. The pattern to notice: the more "per-character" the operation (tokenize, n-grams), the bigger the GPU win — thousands of cores chew thousands of strings at once.

In [ ]:
import json

print(f"{'operation':12s} {'CPU (s)':>9s} {'GPU (s)':>9s} {'speedup':>9s}")
for op, r in BENCH.items():
    if "CPU" in r and "GPU" in r:
        print(f"{op:12s} {r['CPU']:9.3f} {r['GPU']:9.3f} {r['CPU']/r['GPU']:8.1f}x")

# Saved for the slide figure (instructor: download this file after the smoke test)
with open("benchmark_results.json", "w") as f:
    json.dump(BENCH, f, indent=2)

## Act 3 — Twin Hunting: Dedup with MinHash 👯

Raw corpora are full of duplicates and *near-duplicates* (template articles, boilerplate, copy-paste). For LLM training, duplicates = memorization + wasted compute. The problem: comparing every pair among 150k paragraphs = ±11 **billion** comparisons. Not happening.

**MinHash** is the shortcut: each document is reduced to a small *fingerprint*; similar documents produce identical fingerprints with high probability. We just group identical fingerprints — billions of comparisons collapse into one `groupby`.

> Honest note: we use *one* whole fingerprint, so only near-identical duplicates are caught. Production systems (MinHash-LSH) use many fingerprint *bands* so partial similarity is detected too.

In [ ]:
import cudf

# The cuDF minhash API differs slightly across versions — this helper tries both.
def minhash_signature(series, n_hashes=8, width=5):
    try:  # cuDF >= 24.12
        import cupy as cp
        a = cudf.Series(cp.random.RandomState(42).randint(1, 2**31, n_hashes), dtype="uint32")
        b = cudf.Series(cp.random.RandomState(7).randint(0, 2**31, n_hashes), dtype="uint32")
        return series.str.minhash(0, a=a, b=b, width=width)
    except TypeError:  # old API: minhash(seeds, width)
        seeds = cudf.Series(range(n_hashes), dtype="uint32")
        return series.str.minhash(seeds, width=width)

sig = minhash_signature(gdf.str.lower())
# Fingerprint = all minhash values joined into one key string
fingerprint = sig.list.astype("str").str.join("-")
df = cudf.DataFrame({"text": gdf, "fp": fingerprint})
dup_counts = df.groupby("fp").size()
dup_groups = dup_counts[dup_counts > 1]
print(f"{len(dup_groups):,} near-duplicate groups found")

In [ ]:
# Peek at the largest twin group
biggest_fp = dup_groups.sort_values(ascending=False).index[0]
twins = df[df.fp == biggest_fp].text.to_pandas()
for t in twins.head(3):
    print("—", t[:160], "...")

> 🧠 What you just did — minhash + groupby on a GPU — is a mini version of the **fuzzy deduplication** that **NVIDIA NeMo Curator** runs when preparing LLM training corpora. The only difference is scale: us, 150k paragraphs on 1 GPU; Curator, billions of documents on hundreds of GPUs.

## Act 4 — Native cuML: TF-IDF + Sentiment, No Magic 🛠️

In Act 1 the GPU worked silently behind `cuml.accel`. Now we call cuML **directly** — note the API is deliberately a twin of scikit-learn, so everything from notebook 01 transfers 1:1. But before measuring anything, GPU benchmarking has one mandatory ritual: the *warm-up*.

In [ ]:
import cudf
from cuml.feature_extraction.text import TfidfVectorizer as GpuTfidf
from cuml.linear_model import LogisticRegression as GpuLogReg

train_g = cudf.read_csv("smsa_train.tsv", sep="\t", names=["text", "label"])
valid_g = cudf.read_csv("smsa_valid.tsv", sep="\t", names=["text", "label"])
LABEL2ID = {"negative": 0.0, "neutral": 1.0, "positive": 2.0}
ytr = train_g.label.map(LABEL2ID).astype("float32")
yva = valid_g.label.map(LABEL2ID).astype("float32")

# Warm-up: the first CUDA call pays for context init + kernel compilation
# (can take several seconds) — that is NOT the true compute speed.
# Golden rule of GPU benchmarking: never measure the first call.
_w = GpuTfidf(max_features=500).fit_transform(train_g.text.head(256))
_y = cudf.Series([float(i % 2) for i in range(256)], dtype="float32")
GpuLogReg(max_iter=50).fit(_w, _y)
print("GPU is warm — the next measurement is fair")

In [ ]:
import time

t0 = time.perf_counter()
gvec = GpuTfidf(max_features=20_000)
Xtr = gvec.fit_transform(train_g.text)
Xva = gvec.transform(valid_g.text)
gclf = GpuLogReg(max_iter=200)
gclf.fit(Xtr, ytr)
from cuml.metrics import accuracy_score as gpu_accuracy
acc_gpu = gpu_accuracy(yva, gclf.predict(Xva))
t_gpu = time.perf_counter() - t0
print(f"cuML  : accuracy={float(acc_gpu):.4f}  time={t_gpu:.2f}s")

In [ ]:
# Apples-to-apples CPU comparison (same data, without the 20x duplication)
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

train_p = pd.read_csv("smsa_train.tsv", sep="\t", names=["text", "label"])
valid_p = pd.read_csv("smsa_valid.tsv", sep="\t", names=["text", "label"])

t0 = time.perf_counter()
vec = TfidfVectorizer(max_features=20_000)
Xtr = vec.fit_transform(train_p.text)
Xva = vec.transform(valid_p.text)
clf = LogisticRegression(max_iter=200)
clf.fit(Xtr, train_p.label)
acc_cpu = accuracy_score(valid_p.label, clf.predict(Xva))
t_cpu = time.perf_counter() - t0
print(f"sklearn: accuracy={acc_cpu:.4f}  time={t_cpu:.2f}s")
ratio = t_cpu / t_gpu
if ratio >= 1:
    print(f"GPU speedup: {ratio:.1f}x")
else:
    print(f"GPU slower ({ratio:.1f}x) — check: did you skip the warm-up cell above?")

### Native vs accel — when to use which?

| | `cuml.accel` (Act 1) | Native API (Act 4) |
|---|---|---|
| Code changes | zero | swap the imports |
| Control | limited (automatic) | full (dtypes, GPU memory) |
| Best for | quickly migrating old code | new GPU pipelines from scratch |

Two benchmarking lessons from this act: **(1) Never measure the first GPU call** — without a warm-up, cuML can look slower than sklearn (±0.6x on a T4) because the time goes to CUDA initialization, not computation; once warm, the GPU wins by double digits even on data as small as SmSA. **(2) GPUs win at scale** — the bigger the data, the wider the gap; which is exactly why Act 2 used 150k paragraphs.

## Act 5 — Leveling Up: NeMo Curator & the Bridge to LLMs 🌉

Everything you did today is a miniature of a real LLM data pipeline:

| Today (1× T4, 150k paragraphs) | Production (NeMo Curator, hundreds of GPUs, billions of docs) |
|---|---|
| `tokenize`, `normalize` in nvtext | language ID, cleaning, quality filtering |
| MinHash + groupby (Act 3) | distributed MinHash-LSH fuzzy dedup |
| TF-IDF + LogReg (Act 4) | document quality classifiers |

**NVIDIA's NLP library map** — what you've met and will meet:
- **RAPIDS (cuDF/nvtext/cuML)** — today ✅
- **NeMo + NeMo Curator** — LLM training & data curation framework → **Day 06**
- **TensorRT-LLM, Triton** — fast inference → **Day 06**
- **HF Transformers on GPU** — **Day 04**, tomorrow we enter the LLM world

### Exercises
1. Change the Act 2 corpus to 40k articles (`if i >= 40_000`). Does the GPU speedup grow or shrink? Why?
2. In Act 3, reduce the minhash `width` to 3. What happens to the number of duplicate groups, and why?
3. In Act 4, add `ngram_range=(1, 2)` to both vectorizers. Compare accuracy and time.